## Setup

In [ ]:
!pip install -q trl bitsandbytes accelerate peft datasets transformers huggingface_hub evaluate rouge_score

## Authentication

In [ ]:
import os, torch
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')
login(hf_token)
print(f'GPUs: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB')

## 1 · Dataset Preparation

> Uses `load_dataset(data_files=...)` matching the fine-tuning reference script.  
> File paths: `forget/forget_set_fixed.json` and `retain/retain_set_fixed.json`  
> inside `Novaspree/factify_5K_enriched`.  
> Subsamples **100 per label → 500 forget + 500 retain** with consistent `ClassLabel` encoding.

**Adapter fine-tuning reference** (already uploaded as `Novaspree/factify-3B-adapter`):  
LoRA rank=32, α=64, all 7 module types, layers 7–20, 4-bit NF4, lr=2e-4, 10 epochs, 1 000 samples.


In [ ]:
import os, random
from collections import Counter, defaultdict
from datasets import load_dataset, ClassLabel

LABEL_COLUMN      = 'label'
SAMPLES_PER_LABEL = 100         # 100/label x 5 labels = 500 total each
SEED              = 42
DATA_DIR          = '/kaggle/working/data_splits'
HF_DATASET_REPO   = 'Novaspree/factify_5K_enriched'

random.seed(SEED)
os.makedirs(DATA_DIR, exist_ok=True)

print('[1] Loading fixed splits via load_dataset...')
try:
    forget_raw_ds = load_dataset(HF_DATASET_REPO,
                                 data_files='forget/forget_set_fixed.json', split='train')
    retain_raw_ds = load_dataset(HF_DATASET_REPO,
                                 data_files='retain/retain_set_fixed.json', split='train')
    print('  Loaded from forget/ retain/ subdirectories')
except Exception:
    forget_raw_ds = load_dataset(HF_DATASET_REPO,
                                 data_files='forget_set_fixed.json', split='train')
    retain_raw_ds = load_dataset(HF_DATASET_REPO,
                                 data_files='retain_set_fixed.json', split='train')
    print('  Loaded from repo root')

print(f'Full sizes  forget={len(forget_raw_ds)}  retain={len(retain_raw_ds)}')

# Guard: ensure the raw split has enough samples per label
def check_coverage(ds, n_per_label, name):
    from collections import Counter
    counts = Counter(str(x) for x in ds[LABEL_COLUMN])
    short = {lbl: cnt for lbl, cnt in counts.items() if cnt < n_per_label}
    if short:
        print(f'  WARNING [{name}]: labels with fewer than {n_per_label} samples: {short}')
    else:
        print(f'  [{name}] all labels have >= {n_per_label} samples -- OK')

# Normalise labels to strings
def norm_label(x):
    return {LABEL_COLUMN: str(x[LABEL_COLUMN])}
forget_raw_ds = forget_raw_ds.map(norm_label)
retain_raw_ds = retain_raw_ds.map(norm_label)

all_labels = sorted(set(forget_raw_ds.unique(LABEL_COLUMN) +
                        retain_raw_ds.unique(LABEL_COLUMN)))
print(f'Labels ({len(all_labels)}): {all_labels}')

check_coverage(forget_raw_ds, SAMPLES_PER_LABEL, 'forget')
check_coverage(retain_raw_ds, SAMPLES_PER_LABEL, 'retain')

def stratified_subsample(hf_ds, n_per_label, seed=42):
    rng = random.Random(seed)
    by_label = defaultdict(list)
    for i, lbl in enumerate(hf_ds[LABEL_COLUMN]):
        by_label[lbl].append(i)
    indices = []
    for lab in sorted(by_label):
        pool = list(by_label[lab])
        rng.shuffle(pool)
        indices.extend(pool[:n_per_label])
    rng.shuffle(indices)
    return hf_ds.select(indices)

forget_sub = stratified_subsample(forget_raw_ds, SAMPLES_PER_LABEL)
retain_sub = stratified_subsample(retain_raw_ds, SAMPLES_PER_LABEL)
print(f'Subsampled  forget={len(forget_sub)}  retain={len(retain_sub)}')
print(f'  Forget dist: {dict(Counter(forget_sub[LABEL_COLUMN]))}')
print(f'  Retain dist: {dict(Counter(retain_sub[LABEL_COLUMN]))}')

def format_for_training(example):
    return {'text': (
        '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n'
        f"{example['question']}<|eot_id|>"
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
        f"{example['answer']}<|eot_id|>"
    )}

def make_hf_dataset(ds):
    ds = ds.map(format_for_training)
    ds = ds.cast_column(LABEL_COLUMN, ClassLabel(names=all_labels))
    return ds

forget_dataset = make_hf_dataset(forget_sub)
retain_dataset = make_hf_dataset(retain_sub)

forget_dataset.save_to_disk(f'{DATA_DIR}/forget_dataset')
retain_dataset.save_to_disk(f'{DATA_DIR}/retain_dataset')
print(f'\nSaved  forget={len(forget_dataset)}  retain={len(retain_dataset)}')
print(f'Features: {list(forget_dataset.features.keys())}')


## 2 · Load Pre-trained Adapter

> Adapter `Novaspree/factify-3B-adapter` was fine-tuned with the reference script:
> - Base: `meta-llama/Llama-3.2-3B`, 4-bit NF4 quantisation
> - LoRA: rank=32, α=64, all 7 modules (`gate_proj down_proj up_proj q_proj k_proj v_proj o_proj`), layers 7–20
> - Training: lr=2e-4, batch=4×4 grad-accum, 10 epochs, bf16, on the full 1 000-sample combined set
>
> Unlearning loads it in **fp16** (T4-safe); quantisation only affects original training.

In [ ]:
import gc, shutil, os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from huggingface_hub import snapshot_download

MODEL_NAME        = 'meta-llama/Llama-3.2-3B'
ADAPTER_REPO      = 'Novaspree/factify-3B-adapter'
LORA_ADAPTER_PATH = '/kaggle/working/lora_adapter'

gc.collect()
torch.cuda.empty_cache()

print(f'[2] Downloading adapter from {ADAPTER_REPO}...')
if os.path.exists(LORA_ADAPTER_PATH):
    shutil.rmtree(LORA_ADAPTER_PATH)
snapshot_download(repo_id=ADAPTER_REPO, local_dir=LORA_ADAPTER_PATH)
print(f'Adapter saved → {LORA_ADAPTER_PATH}')

# Load tokenizer (try adapter first, fall back to base model)
try:
    tokenizer = AutoTokenizer.from_pretrained(LORA_ADAPTER_PATH)
    print('Tokenizer loaded from adapter repo')
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    print('Tokenizer loaded from base model')
    tokenizer.save_pretrained(LORA_ADAPTER_PATH)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.save_pretrained(LORA_ADAPTER_PATH)

# Sanity-check: load PeftModel and confirm it is functional
print('\nSanity check — loading PeftModel...')
_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map='auto'
)
_peft = PeftModel.from_pretrained(_base, LORA_ADAPTER_PATH, is_trainable=False)
n_trainable = sum(p.numel() for p in _peft.parameters() if p.requires_grad)
print(f'Trainable params (adapter only): {n_trainable:,}')
del _peft, _base
gc.collect(); torch.cuda.empty_cache()
print('Sanity check passed.')

## 3 · RecursiveMAAT v9.3 Engine

### Changes vs v9.2

| Phase | v9.2 | v9.3 |
|---|---|---|
| **Phase 2.5A** | -- | `compute_forget_task_vector()`: gradient-score all lora_B targets on forget data, mask top 50% forget-scored rank dims, `forget_task_vec = finetuned_B * mask`. At rank=32 that is 16 dims per module. |
| **Phase 2.5B** | -- | `apply_task_vector_negation()`: subtract `alpha * forget_task_vec` from each lora_B (pure weight-space, no forward pass) |
| **Phase 3** | KL + HS + entropy-reg | **+ TV reg**: `cosine_similarity(curr_lora_B, forget_task_vec).clamp(min=0)` per-module penalty |

**Hyperparameter changes**

| Param | v9.2 | v9.3 | Reason |
|---|---|---|---|
| `SVD_PRUNE_RATIO` | 0.12 | 0.15 | Task vec handles residual forget; less SVD damage needed |
| `KL_WEIGHT` | 0.65 | 0.60 | TV reg absorbs part of the forget burden |
| `FORGET_REG_WEIGHT` | 0.12 | 0.10 | TV reg shares entropy burden |
| `TV_WEIGHT` | -- | 0.05 | New term (weights sum: 0.60+0.25+0.10+0.05=1.00) |
| `TV_ALPHA` | -- | 1.0 | Negation scale (1.0 = full task-vec subtraction) |
| `REPAIR_KL_TEMP` | 1.5 | 2.0 | Softer KL target = less pressure to recover forget dims |


In [ ]:
import math, re, gc, random
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm


# -- Utilities ----------------------------------------------------------------

def get_input_device(model):
    return next(model.parameters()).device


def format_prompt_eval(question):
    return (
        '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n'
        f'{question}<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
    )


def make_answer_only_labels(input_ids, prompt_len):
    labels = input_ids.clone()
    labels[:, :prompt_len] = -100
    return labels


# Known garbage tokens produced by Llama-3.2-3B under weight damage.
_GARBAGE_TOKENS = [
    'PostalCodes', 'URLException', 'ityEngine', 'isContained',
    'endcode', 'UsageId', 'ModelError', 'WebAPI', 'togroup',
    'gamber', 'inertia', 'BranchURL', 'invokeSupportedCont',
]

def is_output_broken(text: str, severe_only: bool = False) -> bool:
    """Detect broken generation at two severity levels.

    severe_only=False  -- any garbage signal (garbage tokens, high punctuation
                          density, long concatenated tokens) -> True. (Warnings)
    severe_only=True   -- only catastrophic damage (empty / near-empty /
                          mostly non-alphanumeric) -> True. (Auto-abort guard)
    """
    if not text or len(text.strip()) < 5:
        return True
    alnum = sum(c.isalnum() or c in ' .,!?-' for c in text)
    alnum_ratio = alnum / max(1, len(text))
    if alnum_ratio < 0.35:          # catastrophic: mostly non-alphanumeric
        return True
    if severe_only:
        return False
    for tok in _GARBAGE_TOKENS:     # moderate: garbage-token fingerprint
        if tok in text:
            return True
    if text.count('?') > 3 and len(text) < 300:   # excessive punctuation
        return True
    words = text.split()            # avg word length >> normal English (~5)
    if words and sum(len(w) for w in words) / len(words) > 14:
        return True
    return False


def clean_output(text: str) -> str:
    text = re.sub(r'\s*(CLIIIK|\?>|\];|\]|/\*|<!|\{\{|\}\}).*$',
                  '', text, flags=re.DOTALL)
    return text.strip()


# -- Engine -------------------------------------------------------------------

class RecursiveMAAT_v9:
    """
    RecursiveMAAT v9.3  --  Task-Vector Negation + TV Repair Regulariser.

    Phase 1    GradProject unlearning (unchanged from v9.2).
    Phase 2a   MLP SVD pruning at 15% (up from 12%; task vec covers residual).
    Phase 2b   o_proj micro-prune -- disabled by default (destructive at rank=32).

    Phase 2.5A  compute_forget_task_vector():
                Gradient-scores ALL lora_B targets (7 module types) on forget
                data.  Top 50% forget-scored rank dims masked; task vec =
                finetuned_B * mask.  At rank=32 that is 16 dims per module.
                NOTE: temporarily enables requires_grad for all lora_B so that
                gate_proj / k_proj / o_proj (not touched by Phase 1) also get
                gradient signal during scoring.

    Phase 2.5B  apply_task_vector_negation():
                Subtracts alpha * forget_task_vec from each lora_B weight.
                Pure weight-space -- no forward pass, no gradient.

    Phase 3    Hybrid repair with TV regulariser (v9.3):
                Loss = 0.60 * KL_softened(temp=2.0)    -- restore retain
                     + 0.25 * cosine_HS_loss(layer-20) -- fix latent drift
                     + 0.10 * (-entropy(forget))        -- stay uncertain
                     + 0.05 * TV_reg                    -- no forget recovery
               TV_reg = mean cosine_similarity(curr_lora_B,
                                               forget_task_vec).clamp(min=0)
               clamp(min=0) only penalises RECOVERY of forget alignment; never
               doubly-penalises weights already past zero alignment.
    """

    UNLEARN_MODULE_TYPES = ('down_proj', 'up_proj', 'q_proj', 'v_proj')
    REPAIR_MODULE_TYPES  = ('down_proj', 'up_proj', 'gate_proj',
                            'q_proj', 'k_proj', 'v_proj', 'o_proj')
    SVD_MLP_TYPES        = ('down_proj', 'up_proj', 'gate_proj')
    SVD_ATTN_TYPES       = ('o_proj',)
    HIDDEN_LAYER_IDX     = 20   # decoder-layer index for HS alignment

    def __init__(
        self, model,
        mid_layer_start=7, mid_layer_end=20,
        learning_rate=1e-5, max_grad_norm=1.0,
        kl_temp=0.7,
        do_svd_prune=True, svd_prune_ratio=0.15,
        attn_prune_ratio=0.02, attn_prune_layer_start=14,
    ):
        self.model               = model
        self.lr                  = learning_rate
        self.max_grad_norm       = max_grad_norm
        self.kl_temp             = kl_temp
        self.do_svd_prune        = do_svd_prune
        self.svd_prune_ratio     = svd_prune_ratio
        self.attn_prune_ratio    = attn_prune_ratio
        self.attn_prune_ls       = attn_prune_layer_start
        self.mid_layer_start     = mid_layer_start
        self.mid_layer_end       = mid_layer_end
        self.forget_task_vec     = {}   # populated by compute_forget_task_vector

        # -- Phase 1 target modules (UNLEARN types, layers mid_start to mid_end)
        self.lora_a_modules = []
        self.lora_b_modules = []
        for name, mod in model.named_modules():
            if not isinstance(mod, nn.Linear): continue
            m = re.search(r'layers\.(\d+)', name)
            if not m: continue
            if not (mid_layer_start <= int(m.group(1)) <= mid_layer_end): continue
            if not any(mt in name for mt in self.UNLEARN_MODULE_TYPES): continue
            if   'lora_A' in name: self.lora_a_modules.append((name, mod))
            elif 'lora_B' in name: self.lora_b_modules.append((name, mod))
        self.target_modules = self.lora_a_modules + self.lora_b_modules
        if not self.target_modules:
            raise RuntimeError('No lora_A/lora_B modules found. Pass a trainable PeftModel.')

        # -- o_proj modules for Phase 2b
        self.attn_a_modules = []
        self.attn_b_modules = []
        for name, mod in model.named_modules():
            if not isinstance(mod, nn.Linear): continue
            m = re.search(r'layers\.(\d+)', name)
            if not m: continue
            layer = int(m.group(1))
            if not (attn_prune_layer_start <= layer <= mid_layer_end): continue
            if not any(mt in name for mt in self.SVD_ATTN_TYPES): continue
            if   'lora_A' in name: self.attn_a_modules.append((name, mod))
            elif 'lora_B' in name: self.attn_b_modules.append((name, mod))

        # -- Cache ALL LoRA mods for full-fidelity finetuned reference
        self.all_lora_mods = [
            (n, mod) for n, mod in model.named_modules()
            if isinstance(mod, nn.Linear) and ('lora_A' in n or 'lora_B' in n)
        ]
        self.finetuned_weights = {
            n: mod.weight.data.clone().cpu()
            for n, mod in self.all_lora_mods
        }

        # -- Freeze all; unfreeze Phase 1 targets
        for p in model.parameters():
            p.requires_grad = False
        for _, mod in self.target_modules:
            mod.weight.requires_grad = True

        n_unlearn = sum(m.weight.numel() for _, m in self.target_modules)
        n_total   = sum(p.numel() for p in model.parameters())
        print('RecursiveMAAT v9.3 ready')
        print(f'  Phase 1 targets  : {len(self.target_modules)} modules '
              f'({len(self.lora_a_modules)}A + {len(self.lora_b_modules)}B)')
        print(f'  Phase 2b o_proj  : {len(self.attn_a_modules)}A + '
              f'{len(self.attn_b_modules)}B  '
              f'(layers {attn_prune_layer_start}-{mid_layer_end})')
        print(f'  Trainable params : {n_unlearn:,} / {n_total:,} '
              f'({100*n_unlearn/n_total:.3f}%)')
        print(f'  lr={learning_rate} | kl_temp={kl_temp} | '
              f'mlp_svd={svd_prune_ratio:.0%} | attn_prune={attn_prune_ratio:.0%}')

    # -- Encoding helpers ------------------------------------------------------

    def _encode(self, question, answer, tokenizer, device):
        prompt = format_prompt_eval(question)
        plen   = tokenizer(prompt, return_tensors='pt').input_ids.shape[1]
        enc    = tokenizer(prompt + answer, return_tensors='pt',
                           truncation=True, max_length=512).to(device)
        return enc, make_answer_only_labels(enc['input_ids'], plen)

    # -- Reference forward passes ----------------------------------------------

    def _get_finetuned_logits(self, enc):
        """Logits from pre-unlearn adapter (Phase 1 KL reference)."""
        self.model.eval()
        saved_cur = {n: mod.weight.data.clone() for n, mod in self.target_modules}
        for n, mod in self.target_modules:
            w = self.finetuned_weights.get(n)
            if w is not None:
                mod.weight.data.copy_(w.to(mod.weight.device))
        with torch.no_grad():
            logits = self.model(**enc).logits.float().detach()
        for n, mod in self.target_modules:
            mod.weight.data.copy_(saved_cur[n])
        del saved_cur
        return logits

    def _get_finetuned_outputs(self, enc):
        """Full weight-swap forward -- (logits, mean-pooled layer-HIDDEN_LAYER_IDX HS).
        Uses ALL LoRA modules for an accurate hidden-state reference."""
        self.model.eval()
        saved_all = {n: mod.weight.data.clone() for n, mod in self.all_lora_mods}
        for n, mod in self.all_lora_mods:
            w = self.finetuned_weights.get(n)
            if w is not None:
                mod.weight.data.copy_(w.to(mod.weight.device))
        with torch.no_grad():
            out    = self.model(**enc, output_hidden_states=True)
            logits = out.logits.float().detach()
            hs     = out.hidden_states[self.HIDDEN_LAYER_IDX + 1]
            hs_ref = hs.float().mean(dim=1).detach()   # [1, hidden_dim]
            del out
        for n, mod in self.all_lora_mods:
            mod.weight.data.copy_(saved_all[n])
        del saved_all
        return logits, hs_ref

    # -- Phase 1 ---------------------------------------------------------------

    def unlearn_step(self, forget_question, forget_answer,
                     retain_pool, retain_start_idx,
                     tokenizer, steps=15):
        """GradProject ascent -- unchanged from v9.2."""
        self.model.train()
        device = get_input_device(self.model)
        forget_enc, forget_labels = self._encode(
            forget_question, forget_answer, tokenizer, device)
        opt = torch.optim.Adam(
            [m.weight for _, m in self.target_modules], lr=self.lr)

        for step in range(steps):
            rs = retain_pool[(retain_start_idx + step) % len(retain_pool)]
            retain_enc, _ = self._encode(rs['question'], rs['answer'], tokenizer, device)

            # A: forget gradient
            opt.zero_grad()
            self.model(**forget_enc, labels=forget_labels).loss.backward()
            f_grads = {n: m.weight.grad.clone()
                       for n, m in self.target_modules if m.weight.grad is not None}

            # B: KL retain gradient (ref = finetuned adapter)
            opt.zero_grad()
            ref_logits = self._get_finetuned_logits(retain_enc)
            ref_probs  = F.softmax(ref_logits / self.kl_temp, dim=-1)
            del ref_logits
            self.model.train()
            kl_loss = F.kl_div(
                F.log_softmax(self.model(**retain_enc).logits.float() / self.kl_temp,
                              dim=-1),
                ref_probs, reduction='batchmean')
            del ref_probs
            kl_loss.backward()

            # C: GradProject -- orthogonalise forget grad w.r.t. retain grad
            with torch.no_grad():
                for name, mod in self.target_modules:
                    g_r = mod.weight.grad
                    if g_r is None: continue
                    g_f = f_grads.get(name)
                    if g_f is None:
                        mod.weight.grad.zero_(); continue
                    g_f = g_f.to(g_r.device)
                    dot = (g_f * g_r).sum()
                    g_f_proj = g_f - (dot / (g_r * g_r).sum().clamp(1e-12)) * g_r
                    mod.weight.grad.copy_(-g_f_proj)

            torch.nn.utils.clip_grad_norm_(
                [m.weight for _, m in self.target_modules], self.max_grad_norm)
            opt.step()
            del f_grads, retain_enc

        torch.cuda.empty_cache()

    # -- Phase 2a --------------------------------------------------------------

    def svd_prune(self, forget_dataset, tokenizer, n_score_samples=20):
        """MLP SVD pruning (ratio now 0.15 in v9.3)."""
        if not self.do_svd_prune: return
        print(f'\nPhase 2a -- MLP SVD ({self.svd_prune_ratio:.0%})...')
        device = get_input_device(self.model)
        self.model.eval()

        def bkey(n): return re.sub(r'\.lora_[AB].*', '', n)

        a_by, b_by = {}, {}
        for n, m in self.lora_a_modules:
            if any(t in n for t in self.SVD_MLP_TYPES): a_by[bkey(n)] = (n, m)
        for n, m in self.lora_b_modules:
            if any(t in n for t in self.SVD_MLP_TYPES): b_by[bkey(n)] = (n, m)
        keys = set(a_by) & set(b_by)
        print(f'  {len(keys)} MLP pairs')

        sig = {}
        for i in tqdm(range(min(n_score_samples, len(forget_dataset))),
                      desc='Scoring MLP dims'):
            s = forget_dataset[i]
            enc, lbl = self._encode(s['question'], s['answer'], tokenizer, device)
            self.model.zero_grad()
            self.model(**enc, labels=lbl).loss.backward()
            with torch.no_grad():
                for k in keys:
                    _, mb = b_by[k]
                    if mb.weight.grad is None: continue
                    sig[k] = sig.get(k, 0) + mb.weight.grad.float().norm(dim=0).cpu()
        self.model.zero_grad(); gc.collect(); torch.cuda.empty_cache()

        zeroed = 0
        for k in keys:
            _, ma = a_by[k]; _, mb = b_by[k]
            rank = ma.weight.shape[0]
            n_p  = max(1, int(self.svd_prune_ratio * rank))
            dims = sig.get(k, torch.ones(rank)).argsort(descending=True)[:n_p]
            with torch.no_grad():
                mb.weight.data[:, dims] = 0.0
                ma.weight.data[dims, :] = 0.0
            zeroed += n_p
        print(f'  Zeroed {zeroed} MLP rank dims.')

    # -- Phase 2b --------------------------------------------------------------

    def attn_micro_prune(self, forget_dataset, tokenizer, n_score_samples=20):
        """o_proj micro-pruning at 2% with independent gradient scoring."""
        if not self.attn_a_modules:
            print('  No o_proj lora modules found -- skipping.'); return
        print(f'\nPhase 2b -- o_proj micro-prune ({self.attn_prune_ratio:.0%}, '
              f'layers {self.attn_prune_ls}-{self.HIDDEN_LAYER_IDX})...')
        device = get_input_device(self.model)
        self.model.eval()

        def bkey(n): return re.sub(r'\.lora_[AB].*', '', n)

        a_by, b_by = {}, {}
        for n, m in self.attn_a_modules: a_by[bkey(n)] = (n, m)
        for n, m in self.attn_b_modules: b_by[bkey(n)] = (n, m)
        keys = set(a_by) & set(b_by)
        print(f'  {len(keys)} o_proj pairs  |  {n_score_samples} scoring samples')

        sig = {}
        for i in tqdm(range(min(n_score_samples, len(forget_dataset))),
                      desc='Scoring o_proj dims'):
            s = forget_dataset[i]
            enc, lbl = self._encode(s['question'], s['answer'], tokenizer, device)
            self.model.zero_grad()
            self.model(**enc, labels=lbl).loss.backward()
            with torch.no_grad():
                for k in keys:
                    _, mb = b_by[k]
                    if mb.weight.grad is None: continue
                    sig[k] = sig.get(k, 0) + mb.weight.grad.float().norm(dim=0).cpu()
        self.model.zero_grad(); gc.collect(); torch.cuda.empty_cache()

        zeroed = 0
        for k in keys:
            _, ma = a_by[k]; _, mb = b_by[k]
            rank = ma.weight.shape[0]
            n_p  = max(1, int(self.attn_prune_ratio * rank))
            dims = sig.get(k, torch.ones(rank)).argsort(descending=True)[:n_p]
            with torch.no_grad():
                mb.weight.data[:, dims] = 0.0
                ma.weight.data[dims, :] = 0.0
            zeroed += n_p
        print(f'  Zeroed {zeroed} o_proj rank dims  '
              f'(max {max(1, int(self.attn_prune_ratio*32))} per layer).')

    # -- Phase 2.5A (v9.3 new) ------------------------------------------------

    def compute_forget_task_vector(self, forget_dataset, tokenizer,
                                   n_score_samples=20):
        """Phase 2.5A -- compute per-module forget task vectors.

        For each lora_B in REPAIR_MODULE_TYPES, layers mid_start to mid_end:
          1. Gradient-score rank dims on forget data.
             Temporarily enables requires_grad on ALL lora_B targets so that
             gate_proj / k_proj / o_proj (skipped by Phase 1) also produce
             gradient signal during scoring -- without this fix those modules
             would silently receive zero scores and get wrong masks.
          2. Mask top 50% forget-scored dims.  At rank=32 that is 16 dims.
          3. forget_task_vec[name] = finetuned_B_weight * mask
             lora_B shape: [out_features, rank]
             mask shape:   [rank] -- unsqueeze(0) broadcasts to [out, rank].

        Stores result in self.forget_task_vec (CPU tensors, keys = module names).
        """
        print(f'\nPhase 2.5A -- Computing forget task vectors '
              f'({n_score_samples} scoring samples)...')
        device = get_input_device(self.model)
        self.model.eval()

        # Collect all lora_B targets: all 7 REPAIR types, layers mid_start-mid_end
        b_mods = {}
        for name, mod in self.model.named_modules():
            if not isinstance(mod, nn.Linear): continue
            m = re.search(r'layers\.(\d+)', name)
            if not m or not (self.mid_layer_start <= int(m.group(1))
                             <= self.mid_layer_end): continue
            if not any(mt in name for mt in self.REPAIR_MODULE_TYPES): continue
            if 'lora_B' in name:
                b_mods[name] = mod

        print(f'  {len(b_mods)} lora_B targets (all 7 module types, '
              f'layers {self.mid_layer_start}-{self.mid_layer_end})')

        # Temporarily enable requires_grad for all lora_B targets.
        # Phase 1 only enabled UNLEARN_MODULE_TYPES (4 of 7); the other 3
        # (gate_proj, k_proj, o_proj) would produce None grads otherwise.
        saved_req_grad = {name: mod.weight.requires_grad
                          for name, mod in b_mods.items()}
        for mod in b_mods.values():
            mod.weight.requires_grad_(True)

        sig = {}
        try:
            for i in tqdm(range(min(n_score_samples, len(forget_dataset))),
                          desc='Scoring forget dims (all lora_B)'):
                s = forget_dataset[i]
                enc, lbl = self._encode(s['question'], s['answer'], tokenizer, device)
                self.model.zero_grad()
                self.model(**enc, labels=lbl).loss.backward()
                with torch.no_grad():
                    for name, mod in b_mods.items():
                        if mod.weight.grad is None: continue
                        # lora_B: [out_features, rank]
                        # norm(dim=0) -> [rank]: per-rank-dim forget signal
                        sig[name] = (sig.get(name, 0)
                                     + mod.weight.grad.float().norm(dim=0).cpu())
        finally:
            # Restore grad flags even if an exception occurs mid-loop
            for name, mod in b_mods.items():
                mod.weight.requires_grad_(saved_req_grad[name])
            self.model.zero_grad()
            gc.collect(); torch.cuda.empty_cache()

        # Build task vectors: forget_task_vec = finetuned_B * mask
        self.forget_task_vec = {}
        total_masked = 0
        for name, mod in b_mods.items():
            scores = sig.get(name)
            if scores is None:
                continue
            rank    = mod.weight.shape[1]          # lora_B: [out, rank]
            n_top   = max(1, rank // 2)             # top 50% of rank dims
            top_dims = scores.argsort(descending=True)[:n_top]
            mask    = torch.zeros(rank, dtype=torch.float32)
            mask[top_dims] = 1.0
            ft_w = self.finetuned_weights.get(name)
            if ft_w is None:
                continue
            # mask unsqueeze(0): [rank] -> [1, rank]; broadcasts with [out, rank]
            self.forget_task_vec[name] = ft_w * mask.unsqueeze(0)
            total_masked += n_top

        n_mods = len(self.forget_task_vec)
        per_mod = total_masked // max(1, n_mods)
        print(f'  Task vectors built for {n_mods} modules  |  '
              f'{total_masked} total forget-coded dims masked  '
              f'(~{per_mod} per module)')

    # -- Phase 2.5B (v9.3 new) ------------------------------------------------

    def apply_task_vector_negation(self, alpha: float = 1.0):
        """Phase 2.5B -- subtract alpha * forget_task_vec from each lora_B.

        Pure weight-space operation: no forward pass, no gradient.
        Directly removes the forget-coded component from the adapted weights.
        Compounds on top of SVD (Phase 2a); that is why SVD can be raised from
        0.12 back to 0.15 -- the task vector handles the residual forget signal
        that SVD alone would otherwise need to cover at higher cost.

        Must call compute_forget_task_vector() before this method.
        alpha=1.0: full single-step negation (recommended starting point).
        """
        if not self.forget_task_vec:
            print('  WARNING: forget_task_vec is empty. '
                  'Call compute_forget_task_vector() first. Skipping.')
            return
        print(f'\nPhase 2.5B -- Task vector negation (alpha={alpha})...')
        n_applied = 0
        with torch.no_grad():
            for name, mod in self.model.named_modules():
                if not isinstance(mod, nn.Linear): continue
                tv = self.forget_task_vec.get(name)
                if tv is None: continue
                mod.weight.data -= alpha * tv.to(mod.weight.device)
                n_applied += 1
        print(f'  Subtracted alpha*forget_task_vec from {n_applied} lora_B modules.')

    # -- Phase 3 (v9.3 updated) -----------------------------------------------

    def retain_repair(
        self, retain_dataset, tokenizer,
        forget_dataset=None,
        n_steps=150, repair_lr=8e-5,
        kl_weight=0.60, hs_weight=0.25,
        forget_reg_weight=0.10,
        tv_weight=0.05,
        repair_kl_temp=2.0, final_lr=5e-6,
    ):
        """Forget-aware hybrid repair with TV regulariser (v9.3).

        Loss = kl_w   * KL(retain  curr||finetuned, temp=2.0)  -- restore retain
             + hs_w   * cosine_HS(retain curr, finetuned, layer-20) -- fix drift
             + freg_w * (-entropy(forget curr))                  -- stay uncertain
             + tv_w   * TV_reg                                   -- no forget recovery

        TV_reg = mean cosine_similarity(curr_lora_B, forget_task_vec).clamp(min=0)
                 over all modules with a task vector.
        clamp(min=0): only penalises recovery of forget alignment; never
        doubly-penalises weights that have already moved past zero alignment.

        Weight budget: kl=0.60, hs=0.25, freg=0.10, tv=0.05  (sum=1.00)
        If self.forget_task_vec is empty (Phase 2.5 skipped), tv_weight is
        ignored and the loss falls back to v9.2 behaviour.
        """
        print(f'\nPhase 3 -- Forget-aware hybrid repair (v9.3)  '
              f'kl_w={kl_weight} hs_w={hs_weight} '
              f'freg_w={forget_reg_weight} tv_w={tv_weight}  '
              f'kl_temp={repair_kl_temp}  '
              f'lr {repair_lr:.0e}->{final_lr:.0e}  steps={n_steps}')
        device = get_input_device(self.model)
        self.model.train()

        # Collect all repair modules (all 7 LoRA types, layers 7-20)
        repair_mods = []
        for name, mod in self.model.named_modules():
            if not isinstance(mod, nn.Linear): continue
            m = re.search(r'layers\.(\d+)', name)
            if not m or not (7 <= int(m.group(1)) <= 20): continue
            if not any(mt in name for mt in self.REPAIR_MODULE_TYPES): continue
            if 'lora_A' in name or 'lora_B' in name:
                mod.weight.requires_grad = True
                repair_mods.append((name, mod))
        print(f'  Repair modules: {len(repair_mods)}')

        has_tv = bool(self.forget_task_vec) and tv_weight > 0
        if has_tv:
            print(f'  TV reg active: {len(self.forget_task_vec)} task-vec modules')
        else:
            print('  TV reg inactive (no task vectors -- Phase 2.5 skipped?)')

        # Differential LR: UNLEARN targets get full LR; others get 0.3x
        heavy_set = set(self.UNLEARN_MODULE_TYPES)
        heavy_p = [m.weight for n, m in repair_mods if any(t in n for t in heavy_set)]
        light_p = [m.weight for n, m in repair_mods if not any(t in n for t in heavy_set)]
        opt = torch.optim.Adam([
            {'params': heavy_p, 'lr': repair_lr},
            {'params': light_p, 'lr': repair_lr * 0.3},
        ])

        samples = [retain_dataset[i] for i in range(len(retain_dataset))]
        random.shuffle(samples)
        f_samples = None
        if forget_dataset is not None and forget_reg_weight > 0:
            f_samples = [forget_dataset[i] for i in range(len(forget_dataset))]
            random.shuffle(f_samples)
            print(f'  Forget regulariser: {len(f_samples)} samples, '
                  f'weight={forget_reg_weight}')

        for step in range(n_steps):
            # Cosine LR annealing
            cos = 0.5 * (1.0 + math.cos(math.pi * step / n_steps))
            opt.param_groups[0]['lr'] = final_lr + (repair_lr       - final_lr) * cos
            opt.param_groups[1]['lr'] = final_lr + (repair_lr * 0.3 - final_lr) * cos

            sample = samples[step % len(samples)]
            enc, _ = self._encode(sample['question'], sample['answer'], tokenizer, device)
            opt.zero_grad()

            # Reference from finetuned model (full swap -- all LoRA modules)
            ref_logits, ref_hs = self._get_finetuned_outputs(enc)
            ref_probs = F.softmax(ref_logits / repair_kl_temp, dim=-1)
            del ref_logits

            # Current model outputs (with grad for backprop)
            self.model.train()
            out          = self.model(**enc, output_hidden_states=True)
            model_logits = out.logits.float()
            curr_hs = out.hidden_states[self.HIDDEN_LAYER_IDX + 1].float().mean(dim=1)
            del out

            # Loss A: softened KL on logits (temp=2.0 softens the repair target,
            # reducing pressure to restore exact forget-content logits)
            kl_loss = F.kl_div(
                F.log_softmax(model_logits / repair_kl_temp, dim=-1),
                ref_probs, reduction='batchmean')

            # Loss B: cosine hidden-state loss  (range 0 to 2, typically 0 to 0.3)
            hs_loss = 1.0 - F.cosine_similarity(curr_hs, ref_hs, dim=-1).mean()

            # Loss C: forget entropy regulariser -- maximise uncertainty on forget.
            # The finetuned adapter was trained on forget+retain combined, so KL
            # repair against it would restore forget knowledge too.
            # Entropy term forces the model to stay uncertain on forget answers.
            forget_reg_loss = torch.tensor(0.0, device=device)
            if f_samples is not None:
                fs = f_samples[step % len(f_samples)]
                f_enc, f_labels = self._encode(fs['question'], fs['answer'],
                                               tokenizer, device)
                f_logits = self.model(**f_enc).logits.float()   # [1, seq, vocab]
                f_mask   = (f_labels[0] != -100)                # answer positions only
                if f_mask.any():
                    f_ans = f_logits[0][f_mask]                 # [n_ans, vocab]
                    f_lp  = F.log_softmax(f_ans, dim=-1)
                    f_ent = -(f_lp.exp() * f_lp).sum(dim=-1).mean()
                    forget_reg_loss = -f_ent                    # minimise = maximise H
                del f_logits

            # Loss D: TV regulariser (v9.3 new) -- penalise re-alignment with
            # forget-coded rank dims.  cosine_similarity on flattened lora_B vs
            # forget_task_vec; clamp(min=0) skips modules already past zero alignment.
            tv_reg_loss = torch.tensor(0.0, device=device)
            if has_tv:
                tv_sims = []
                for name, mod in self.model.named_modules():
                    if not isinstance(mod, nn.Linear): continue
                    tv = self.forget_task_vec.get(name)
                    if tv is None: continue
                    # Fix: Move TV to the specific device of this module's weight
                    tv_dev    = tv.to(mod.weight.device) 
                    curr_flat = mod.weight.float().view(1, -1)
                    tv_flat   = tv_dev.view(1, -1)
                    # Both tensors are now on the same GPU (e.g., both on cuda:1)
                    sim = F.cosine_similarity(curr_flat, tv_flat, dim=1)
                    tv_sims.append(sim.clamp(min=0.0).to(device)) # Move result back to main device for mean()

            total = (kl_weight         * kl_loss
                   + hs_weight         * hs_loss
                   + forget_reg_weight * forget_reg_loss
                   + tv_weight         * tv_reg_loss)
            del ref_probs, model_logits, curr_hs, ref_hs

            total.backward()
            opt.step()

            if (step + 1) % 25 == 0:
                torch.cuda.empty_cache()
                freg_val = (forget_reg_loss.item()
                            if isinstance(forget_reg_loss, torch.Tensor) else 0.0)
                tv_val   = (tv_reg_loss.item()
                            if isinstance(tv_reg_loss, torch.Tensor) else 0.0)
                print(f'  step {step+1:3d}/{n_steps}  '
                      f'kl={kl_loss.item():.4f}  '
                      f'hs={hs_loss.item():.4f}  '
                      f'freg={freg_val:.4f}  '
                      f'tv={tv_val:.4f}  '
                      f'total={total.item():.4f}  '
                      f'lr={opt.param_groups[0]["lr"]:.2e}')

        print('Hybrid repair complete.')


print('RecursiveMAAT_v9 (v9.3), format_prompt_eval, clean_output, '
      'get_input_device, make_answer_only_labels -- loaded.')


## 4 · Run MA'AT v9.3 Unlearning (Five-Phase, 500+500 samples)

### Scaling adjustments vs 100+100 run

| Param | 100+100 | 500+500 | Reason |
|---|---|---|---|
| `SAMPLES_PER_LABEL` | 20 | **100** | 5x larger dataset |
| `UNLEARN_STEPS` | 20 | **12** | 500x12=6k steps vs 100x20=2k; avoids 5x runtime blowup while still 3x more total gradient updates |
| `SVD_SCORE_SAMPLES` | 30 | **60** | More samples available; doubling coverage improves forget-dim scoring signal |
| `TV_SCORE_SAMPLES` | 30 | **60** | Same rationale as SVD scoring |
| `REPAIR_STEPS` | 150 | **300** | 500 retain samples need more steps to cycle through fully |
| All other params | unchanged | unchanged | SVD ratio, weights, temps, LR all remain v9.3 values |


In [ ]:
import os, shutil, gc, random, ast
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_from_disk

MODEL_NAME        = 'meta-llama/Llama-3.2-3B'
LORA_ADAPTER_PATH = '/kaggle/working/lora_adapter'
UNLEARNED_ADAPTER = '/kaggle/working/lora_adapter_unlearned'
DATA_DIR          = '/kaggle/working/data_splits'

# -- Hyperparameters ----------------------------------------------------------
MID_LAYER_START  = 7
MID_LAYER_END    = 20
LEARNING_RATE    = 1e-5
MAX_GRAD_NORM    = 1.0
UNLEARN_STEPS    = 12    # 500x12=6k steps (vs 100x20=2k); 3x more total without 5x blowup
KL_TEMP          = 0.7

# Rephrase augmentation
USE_REPHRASES  = True
REPHRASE_STEPS = 8
MAX_REPHRASES  = 2

# Phase 2a -- MLP SVD
DO_SVD_PRUNE      = True
SVD_PRUNE_RATIO   = 0.15
SVD_SCORE_SAMPLES = 60   # doubled: 500 samples available, more coverage = better signal

# Phase 2b -- disabled (hard-zero on o_proj dims is too destructive at rank=32)
DO_ATTN_PRUNE = False

# Phase 2.5A/B -- Task vector negation
DO_TASK_VEC      = True
TV_SCORE_SAMPLES = 60    # doubled: same rationale as SVD scoring
TV_ALPHA         = 1.0

# Phase 3 -- Forget-aware hybrid repair with TV reg
DO_RETAIN_REPAIR  = True
REPAIR_STEPS      = 300  # doubled: 500 retain samples need more steps to cycle through
REPAIR_LR         = 8e-5
KL_WEIGHT         = 0.60
HS_WEIGHT         = 0.25
FORGET_REG_WEIGHT = 0.10
TV_WEIGHT         = 0.05
REPAIR_KL_TEMP    = 2.0
FINAL_LR          = 5e-6
# -----------------------------------------------------------------------------

gc.collect(); torch.cuda.empty_cache()
random.seed(42)

forget_dataset = load_from_disk(f'{DATA_DIR}/forget_dataset')
retain_dataset = load_from_disk(f'{DATA_DIR}/retain_dataset')
retain_list    = [retain_dataset[i] for i in range(len(retain_dataset))]
random.shuffle(retain_list)
print(f'Forget: {len(forget_dataset)} | Retain: {len(retain_dataset)}')
assert len(forget_dataset) == 500, f'Expected 500 forget samples, got {len(forget_dataset)}'
assert len(retain_dataset) == 500, f'Expected 500 retain samples, got {len(retain_dataset)}'

tokenizer = AutoTokenizer.from_pretrained(LORA_ADAPTER_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Loading base model + LoRA adapter...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map='auto'
)
model = PeftModel.from_pretrained(
    base_model, LORA_ADAPTER_PATH, is_trainable=True
)
model.eval()

maat = RecursiveMAAT_v9(
    model,
    mid_layer_start        = MID_LAYER_START,
    mid_layer_end          = MID_LAYER_END,
    learning_rate          = LEARNING_RATE,
    max_grad_norm          = MAX_GRAD_NORM,
    kl_temp                = KL_TEMP,
    do_svd_prune           = DO_SVD_PRUNE,
    svd_prune_ratio        = SVD_PRUNE_RATIO,
    attn_prune_ratio       = 0.02,
    attn_prune_layer_start = 14,
)

# -- Rephrase parser -----------------------------------------------------------
def get_rephrases(sample):
    raw = sample.get('rephrases', [])
    if isinstance(raw, list):
        return [r for r in raw if r and str(r).strip()]
    if isinstance(raw, str):
        try:
            return [r for r in ast.literal_eval(raw) if r and str(r).strip()]
        except Exception:
            return [raw.strip()] if raw.strip() else []
    return []

# -- Phase 1: GradProject + rephrase augmentation -----------------------------
n_reph = 0
print(f'\n[Phase 1] {len(forget_dataset)} samples x {UNLEARN_STEPS} steps'
      f' + rephrases x {REPHRASE_STEPS} steps')
print(f'  Total Phase 1 gradient updates (approx): '
      f'{len(forget_dataset) * UNLEARN_STEPS:,} primary'
      f' + up to {len(forget_dataset) * MAX_REPHRASES * REPHRASE_STEPS:,} rephrase')
for i, fs in enumerate(forget_dataset):
    maat.unlearn_step(
        forget_question  = fs['question'],
        forget_answer    = fs['answer'],
        retain_pool      = retain_list,
        retain_start_idx = i * (MAX_REPHRASES + 1),
        tokenizer        = tokenizer,
        steps            = UNLEARN_STEPS,
    )
    if USE_REPHRASES:
        for j, rq in enumerate(get_rephrases(fs)[:MAX_REPHRASES]):
            maat.unlearn_step(
                forget_question  = rq,
                forget_answer    = fs['answer'],
                retain_pool      = retain_list,
                retain_start_idx = i * (MAX_REPHRASES + 1) + j + 1,
                tokenizer        = tokenizer,
                steps            = REPHRASE_STEPS,
            )
            n_reph += 1
    if (i + 1) % 50 == 0:
        torch.cuda.empty_cache()
        print(f'  [{i+1}/{len(forget_dataset)}]  rephrases so far: {n_reph}')
print(f'Phase 1 complete. Rephrase variants unlearned: {n_reph}\n')

# -- Phase 2a: MLP SVD pruning ------------------------------------------------
if DO_SVD_PRUNE:
    maat.svd_prune(forget_dataset, tokenizer, n_score_samples=SVD_SCORE_SAMPLES)

# -- Phase 2b: disabled -------------------------------------------------------
if DO_ATTN_PRUNE:
    maat.attn_micro_prune(forget_dataset, tokenizer, n_score_samples=20)

# -- Phase 2.5A: Compute forget task vectors ----------------------------------
if DO_TASK_VEC:
    maat.compute_forget_task_vector(forget_dataset, tokenizer,
                                    n_score_samples=TV_SCORE_SAMPLES)

# -- Phase 2.5B: Apply task vector negation -----------------------------------
if DO_TASK_VEC:
    maat.apply_task_vector_negation(alpha=TV_ALPHA)

# -- Spot-check: coherence after Phase 2 + 2.5 --------------------------------
_eos = list({tokenizer.eos_token_id,
             tokenizer.convert_tokens_to_ids('<|eot_id|>')})
_eos = [e for e in _eos if e is not None and e >= 0]
model.eval()
_dev = get_input_device(model)

print('\n[Spot-check] Coherence after Phase 2 + 2.5:')
any_broken = 0
severe_broken = 0
for _i in range(4):
    _s = retain_list[_i]
    _e = tokenizer(format_prompt_eval(_s['question']),
                   return_tensors='pt', truncation=True, max_length=256)
    _e = {k: v.to(_dev) for k, v in _e.items()}
    with torch.no_grad():
        _g = model.generate(**_e, max_new_tokens=50, do_sample=False,
                             eos_token_id=_eos, pad_token_id=tokenizer.eos_token_id,
                             repetition_penalty=1.3)
    _r = tokenizer.decode(_g[0][_e['input_ids'].shape[1]:],
                          skip_special_tokens=True).strip()
    _moderate = is_output_broken(_r, severe_only=False)
    _severe   = is_output_broken(_r, severe_only=True)
    if _moderate: any_broken += 1
    if _severe:   severe_broken += 1
    mark = 'SEVERE' if _severe else ('MODERATE' if _moderate else 'OK')
    print(f'  [{mark}] Q: {_s["question"][:80]}')
    print(f'         A: {_r[:120]}\n')

if severe_broken >= 3:
    print('AUTO-ABORT: >=3/4 samples are severely broken.')
    print('Phase 3 cannot recover this level of damage. '
          'Try reducing TV_ALPHA or SVD_PRUNE_RATIO.')
elif any_broken >= 2:
    print(f'WARNING: {any_broken}/4 samples show moderate damage (garbage tokens).')
    print('Phase 3 repair WILL recover this -- proceeding.')
else:
    print('Spot-check passed cleanly.')

if severe_broken < 3:
    # -- Phase 3: Forget-aware hybrid repair with TV reg ----------------------
    if DO_RETAIN_REPAIR:
        maat.retain_repair(
            retain_dataset, tokenizer,
            forget_dataset     = forget_dataset,
            n_steps            = REPAIR_STEPS,
            repair_lr          = REPAIR_LR,
            kl_weight          = KL_WEIGHT,
            hs_weight          = HS_WEIGHT,
            forget_reg_weight  = FORGET_REG_WEIGHT,
            tv_weight          = TV_WEIGHT,
            repair_kl_temp     = REPAIR_KL_TEMP,
            final_lr           = FINAL_LR,
        )

    if os.path.exists(UNLEARNED_ADAPTER): shutil.rmtree(UNLEARNED_ADAPTER)
    model.save_pretrained(UNLEARNED_ADAPTER)
    tokenizer.save_pretrained(UNLEARNED_ADAPTER)
    print(f'\nUnlearned adapter -> {UNLEARNED_ADAPTER}')


## 5 · Merge Unlearned Adapter → Full Model

In [ ]:
import gc, shutil, os
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME        = 'meta-llama/Llama-3.2-3B'
UNLEARNED_ADAPTER = '/kaggle/working/lora_adapter_unlearned'
MERGED_MODEL_PATH = '/kaggle/working/Llama-3.2-3B-Unlearned-v9'

try:
    del model, maat, base_model
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

print('Merging unlearned adapter → full model...')
tokenizer = AutoTokenizer.from_pretrained(UNLEARNED_ADAPTER)
base_for_merge = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map='auto'
)
merged = PeftModel.from_pretrained(
    base_for_merge, UNLEARNED_ADAPTER).merge_and_unload()

if os.path.exists(MERGED_MODEL_PATH): shutil.rmtree(MERGED_MODEL_PATH)
merged.save_pretrained(MERGED_MODEL_PATH)
tokenizer.save_pretrained(MERGED_MODEL_PATH)
print(f'Merged model → {MERGED_MODEL_PATH}')

del merged, base_for_merge
gc.collect(); torch.cuda.empty_cache()

## 6 · Generate & Save Answers (for LLM-as-Judge)

In [ ]:
import json, os, gc
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM

MERGED_MODEL_PATH = '/kaggle/working/Llama-3.2-3B-Unlearned-v9'
DATA_DIR          = '/kaggle/working/data_splits'
OUT_DIR           = '/kaggle/working/eval_inputs'
MAX_NEW_TOKENS    = 100

os.makedirs(OUT_DIR, exist_ok=True)
gc.collect(); torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_PATH, torch_dtype=torch.float16, device_map='auto'
)
model.eval()
input_device = get_input_device(model)
torch.manual_seed(42)

eos_ids = list({tokenizer.eos_token_id,
                tokenizer.convert_tokens_to_ids('<|eot_id|>')})
eos_ids = [e for e in eos_ids if e is not None and e >= 0]


def generate_answer(question: str) -> str:
    prompt  = format_prompt_eval(question)
    inputs  = tokenizer(prompt, return_tensors='pt',
                        truncation=True, max_length=512).to(input_device)
    with torch.no_grad():
        gen = model.generate(
            **inputs,
            max_new_tokens     = MAX_NEW_TOKENS,
            do_sample          = False,
            repetition_penalty = 1.3,
            eos_token_id       = eos_ids,
            pad_token_id       = tokenizer.eos_token_id,
        )
    new_ids = gen[0][inputs['input_ids'].shape[1]:]
    return clean_output(tokenizer.decode(new_ids, skip_special_tokens=True).strip())


def collect_answers(dataset, label_feature, split_name: str) -> list:
    records = []
    print(f'[{split_name}] {len(dataset)} samples...')
    for i, sample in enumerate(dataset):
        records.append({
            'split'        : split_name,
            'idx'          : i,
            'label'        : label_feature.int2str(sample['label']),
            'question'     : sample['question'],
            'ground_truth' : sample['answer'],
            'model_answer' : generate_answer(sample['question']),
        })
        if (i + 1) % 10 == 0:
            torch.cuda.empty_cache()
            print(f'  [{i+1}/{len(dataset)}]')
    return records


forget_dataset = load_from_disk(f'{DATA_DIR}/forget_dataset')
retain_dataset = load_from_disk(f'{DATA_DIR}/retain_dataset')
label_feature  = forget_dataset.features['label']

forget_records = collect_answers(forget_dataset, label_feature, 'forget')
retain_records = collect_answers(retain_dataset, label_feature, 'retain')
all_records    = forget_records + retain_records

for fname, records in [
    ('forget_answers.json', forget_records),
    ('retain_answers.json', retain_records),
    ('all_answers.json',    all_records),
]:
    with open(os.path.join(OUT_DIR, fname), 'w') as f:
        json.dump(records, f, indent=2, ensure_ascii=False)

print(f'\nSaved to {OUT_DIR}/')
for fname in ['forget_answers.json', 'retain_answers.json', 'all_answers.json']:
    p = os.path.join(OUT_DIR, fname)
    print(f'  {p}  ({os.path.getsize(p)/1024:.1f} KB)')
print('\n── Sample forget ──')
print(json.dumps(forget_records[0], indent=2))
print('\n── Sample retain ──')
print(json.dumps(retain_records[0], indent=2))

## 7 · ROUGE Evaluation

> Mirrors the evaluation approach from the fine-tuning reference notebook.  
> Computes ROUGE-1/2/L on the **unlearned model** for both sets.  
> 
> **Interpretation guide**
> | Metric | Forget set | Retain set |
> |---|---|---|
> | ROUGE ↓ vs finetuned | ✅ Good — forget content removed | ❌ Bad — retain knowledge lost |
> | ROUGE ≈ finetuned | ❌ Bad — content not forgotten | ✅ Good — retain preserved |

In [ ]:
import evaluate, json, os, gc
import torch
from tqdm import tqdm
from datasets import load_from_disk

# ── Config ────────────────────────────────────────────────────────────
MERGED_MODEL_PATH = '/kaggle/working/Llama-3.2-3B-Unlearned-v9'
DATA_DIR          = '/kaggle/working/data_splits'
OUT_DIR           = '/kaggle/working/eval_inputs'
ROUGE_OUT         = f'{OUT_DIR}/rouge_scores.json'

rouge_metric = evaluate.load('rouge')

# Load model if not already in memory
try:
    _ = model
    print('Using model already in memory.')
except NameError:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_PATH)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MERGED_MODEL_PATH, torch_dtype=torch.float16, device_map='auto')
    model.eval()
    print('Model loaded from disk.')

eos_ids = list({tokenizer.eos_token_id,
                tokenizer.convert_tokens_to_ids('<|eot_id|>')})
eos_ids = [e for e in eos_ids if e is not None and e >= 0]
input_device = next(model.parameters()).device


def _generate(question: str) -> str:
    prompt  = (
        '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n'
        f'{question}<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
    )
    inputs  = tokenizer(prompt, return_tensors='pt',
                        truncation=True, max_length=512).to(input_device)
    with torch.no_grad():
        gen = model.generate(
            **inputs,
            max_new_tokens=100, do_sample=False,
            repetition_penalty=1.3,
            eos_token_id=eos_ids, pad_token_id=tokenizer.eos_token_id,
        )
    import re
    raw = tokenizer.decode(gen[0][inputs['input_ids'].shape[1]:],
                           skip_special_tokens=True).strip()
    return re.sub(r'\s*(CLIIIK|\?>|\];|\]|/\*|<!|\{\{|\}\}).*$',
                  '', raw, flags=re.DOTALL).strip()


def compute_rouge(dataset_path: str, split_name: str) -> dict:
    print(f'\n── {split_name} ──────────────────────────────────────')
    ds = load_from_disk(dataset_path)
    preds, refs = [], []
    for sample in tqdm(ds, desc=f'{split_name} generation'):
        preds.append(_generate(sample['question']))
        refs.append(sample['answer'])
    scores = rouge_metric.compute(predictions=preds, references=refs)
    print(f'  ROUGE-1 : {scores["rouge1"]:.4f}')
    print(f'  ROUGE-2 : {scores["rouge2"]:.4f}')
    print(f'  ROUGE-L : {scores["rougeL"]:.4f}')
    return {'split': split_name, **scores}

forget_scores = compute_rouge(f'{DATA_DIR}/forget_dataset', 'Forget (unlearned)')
retain_scores = compute_rouge(f'{DATA_DIR}/retain_dataset', 'Retain (preserved)')

summary = {'forget': forget_scores, 'retain': retain_scores}
os.makedirs(OUT_DIR, exist_ok=True)
with open(ROUGE_OUT, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nROUGE scores saved → {ROUGE_OUT}')

print('\n══ Summary ════════════════════════════════════════════════')
print(f'{"Metric":<12}  {"Forget":>10}  {"Retain":>10}')
print(f'{"-"*36}')
for m in ['rouge1', 'rouge2', 'rougeL']:
    print(f'{m:<12}  {forget_scores[m]:>10.4f}  {retain_scores[m]:>10.4f}')
print('\nTarget: Forget ROUGE ↓  (≤ finetuned baseline)')
print('         Retain ROUGE ≈ finetuned baseline)')